In [ ]:
import os
from openai import OpenAI
# Importar bibliotecas necesarias para memoria
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
import gradio as gr

# Configurar cliente OpenAI
try:
    llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o-mini",
        streaming=True,
        temperature=0.5
    )

    print("✓ Modelo configurado para experimentos de memoria")
    print(f"Modelo: {llm.model_name}")

except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno")

# Almacén de historiales
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# Crear conversación
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en todos las arias de TCG ya sea de pokemon, yugioh, riftbound, etc., que controla sobre valor de sobres y cartas individuales. Respondes con información precisa y actualizada sobre el valor de cartas y sobres, así como consejos para coleccionistas y jugadores."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def chat_interactivo():
    print("=== CHAT INTERACTIVO CON MEMORIA ===")
    print("Escribe 'salir' para terminar la conversación\n")

    session_id = "chat_session"

    while True:
        try:
            # Pedir input del usuario
            user_input = input("👤 Tú: ").strip()

            if user_input.lower() in ['salir', 'exit', 'quit']:
                print("¡Hasta luego!")
                break

            if not user_input:
                continue

            # Mostrar respuesta con streaming
            print("🤖 Asistente: ", end="", flush=True)
            
            full_response = ""
            
            # Usar stream() para obtener respuestas en tiempo real
            for chunk in conversation.stream(
                {"input": user_input},
                config={"configurable": {"session_id": session_id}}
            ):
                # Extraer el contenido del chunk
                if hasattr(chunk, 'content'):
                    content = chunk.content
                else:
                    content = str(chunk)
                
                print(content, end="", flush=True)
                full_response += content
            
            print("\n")

        except KeyboardInterrupt:
            print("\n¡Conversación interrumpida!")
            break
        except Exception as e:
            print(f"Error: {e}")
            break

# Ejecutar chat interactivo
if __name__ == "__main__":
    
    chat_interactivo()

✓ Modelo configurado para experimentos de memoria
Modelo: gpt-4o-mini
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


✓ Modelo configurado para experimentos de memoria
Modelo: gpt-4o-mini
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


***Interfaz con GRADIO***

In [ ]:
import os
from openai import OpenAI
# Importar bibliotecas necesarias para memoria
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
import gradio as gr

# Configurar cliente OpenAI
try:
    llm = ChatOpenAI(
        base_url=os.getenv("OPENAI_BASE_URL"),
        api_key=os.getenv("GITHUB_TOKEN"),
        model="gpt-4o-mini",
        streaming=True,
        temperature=0.5
    )

    print("✓ Modelo configurado para experimentos de memoria")
    print(f"Modelo: {llm.model_name}")

except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica las variables de entorno")

# Almacén de historiales
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# Crear conversación
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un experto en todos las arias de TCG ya sea de pokemon, yugioh, riftbound, etc., que controla sobre valor de sobres y cartas individuales. Respondes con información precisa y actualizada sobre el valor de cartas y sobres, así como consejos para coleccionistas y jugadores."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

conversation = RunnableWithMessageHistory(
    prompt | llm,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

def respond(message, history):
    response = conversation.invoke(
        {"input": message}, 
        config={"configurable": {"session_id": "gradio_session"}}
    )
    return response.content

# Crear interfaz Gradio
demo = gr.ChatInterface(
    respond,
    title="Asistente TCG Experto",
    description="Pregunta sobre valores de cartas y sobres de TCG como Pokémon, Yu-Gi-Oh, etc."
)

# Lanzar la interfaz
demo.launch()